# S3 event STAR tearsheet

Loads the frozen `event` STAR stack, runs research-IS + sealed OOS backtests,
writes PDF tearsheets under `04_backtest/s3_fx_trend/artifacts/`.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s3_fx_trend.report import load_star_stack, require_star
from backtest.s3_fx_trend.research import (
    RESEARCH_IS_END_S3,
    config_from_stack,
    star_stack_path,
)
from backtest.s3_fx_trend.runner import run_s3_backtest
from backtest.s3_fx_trend.tearsheet import write_tearsheet_pdf
from data.processing.s3_fx_price_panel import price_panel_path, s3_data_dir

SLEEVE = "event"
DATA_DIR = s3_data_dir(ROOT)
ARTIFACTS = os.path.join(ROOT, "04_backtest", "s3_fx_trend", "artifacts")
os.makedirs(ARTIFACTS, exist_ok=True)
STACK_PATH = star_stack_path(SLEEVE)
stack = load_star_stack(STACK_PATH)
print(STACK_PATH)
print(stack)


## 1. Data


In [ ]:
path = price_panel_path("1d", data_dir=DATA_DIR)
if not os.path.isfile(path):
    warnings.warn(f"missing {path}")
    panel = pd.DataFrame()
else:
    panel = pd.read_parquet(path)
    panel["date"] = pd.to_datetime(panel["date"])

rates_path = os.path.join(DATA_DIR, "g10_policy_rates.parquet")
rates = pd.read_parquet(rates_path) if os.path.isfile(rates_path) else pd.DataFrame()
reer_path = os.path.join(DATA_DIR, "bis_reer_monthly.parquet")
reer = pd.read_parquet(reer_path) if os.path.isfile(reer_path) else pd.DataFrame()

if not panel.empty:
    panel_is = panel.loc[panel["date"] <= pd.Timestamp(RESEARCH_IS_END_S3)].copy()
    panel_oos = panel.loc[panel["date"] > pd.Timestamp(RESEARCH_IS_END_S3)].copy()
else:
    panel_is = panel_oos = panel
print("IS", len(panel_is), "OOS", len(panel_oos))


## 2. Run IS + sealed OOS


In [ ]:
try:
    cfg = config_from_stack(stack)
except Exception as exc:
    warnings.warn(f"STAR incomplete — cannot build config yet: {exc!r}")
    cfg = None

if cfg is None or panel_is.empty:
    print("Skip backtest until STAR keys are frozen and panels exist.")
else:
    res_is = run_s3_backtest(
        panel_is, cfg,
        rates_df=rates if not rates.empty else None,
        reer_df=reer if not reer.empty else None,
    )
    display(pd.Series(res_is.metrics, name="IS"))
    is_pdf = os.path.join(ARTIFACTS, f"s3_{SLEEVE}_is_tearsheet.pdf")
    write_tearsheet_pdf(res_is, is_pdf, title=f"S3 {SLEEVE} IS", sleeve=SLEEVE)
    print("wrote", is_pdf)

    if panel_oos.empty:
        print("No OOS rows")
    else:
        res_oos = run_s3_backtest(
            panel_oos, cfg,
            rates_df=rates if not rates.empty else None,
            reer_df=reer if not reer.empty else None,
        )
        display(pd.Series(res_oos.metrics, name="sealed_OOS"))
        oos_pdf = os.path.join(ARTIFACTS, f"s3_{SLEEVE}_oos_tearsheet.pdf")
        write_tearsheet_pdf(res_oos, oos_pdf, title=f"S3 {SLEEVE} sealed OOS", sleeve=SLEEVE)
        print("wrote", oos_pdf)
